# 08 — PCC di backbone kedua (ResNet-50 UMTTA), TANPA GPU

Cache logit dari notebook 07 membuat semuanya di sini murni CPU.

**Yang diuji, dan kenapa tiga keluarga φ, bukan satu.** Backbone di sini adalah
torchvision ResNet-50 — model yang **sama** dengan sumber φ kepala pada hasil
ImageNet kita. Di CCC itu tidak masalah (skornya SimCLRv2+probe, jadi φ eksogen),
tetapi di sini eksogenitasnya hilang. Menjalankan apa adanya akan menguji klaim yang
**berbeda** tanpa menyadarinya.

| φ | sumber | menguji |
|---|---|---|
| `head_rn50` | **model yang sama** dengan skornya | setting deployment realistis |
| `head_vit` | ViT-B/16 — model **berbeda** | **eksogen** — replikasi klaim ImageNet |
| `output` | matriks skor | kontrol sirkular |

Deskriptor kepala bekerja pada **kosinus antar baris** `w_y`, jadi dimensi fitur
tidak perlu cocok: ResNet-50 (1000, 2048) dan ViT-B/16 (1000, 768) sama-sama sah.

**Skornya harus memakai suhu Platt mereka.** Cache menyimpan logit, dan manifest
menyimpan temperature per metode. Memakai softmax mentah akan menghasilkan skor yang
BUKAN skor mereka, dan perbandingannya tidak sah.

## 1. Config

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/pcc'
CACHE = DRIVE_ROOT + '/umtta/imagenet_resnet50'
REPO_URL = ''            # isi kalau perlu clone; kosong = sudah ada di /content
REPO_DIR = 'foundation-cp'
WORK = '/content/umtta_npy'

METHODS = ('baseline', 'tta_avg', 'tta_learned', 'umtta')
SEEDS   = (0, 1, 2)
ALPHAS  = (0.10, 0.05)
N_CALS  = (25, 50)
HELDOUT_FRAC = 0.30
SEED = 42
print('metode', len(METHODS), '| seed', len(SEEDS), '| alpha', ALPHAS)

## 2. Drive, repo, dan verifikasi cache

In [ ]:
import os, glob, json, subprocess, sys, time
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
print('cwd', os.getcwd())

seeds_found = sorted(os.path.basename(d) for d in glob.glob(CACHE + '/seed_*'))
assert seeds_found, 'cache tidak ada di ' + CACHE
print('seed di cache:', seeds_found)
man = json.load(open(CACHE + '/' + seeds_found[0] + '/manifest.json'))
print('kunci manifest:', sorted(man))
print('cal_size', man.get('cal_size'), '| test_size', man.get('test_size'))
print('temperature   :', man.get('temperature'))
print('temperatures  :', man.get('temperatures'))

## 3. Kepala klasifier → `.npy`, DUA model

ResNet-50 (sama dengan backbone) dan ViT-B/16 (berbeda). Keduanya hanya unduhan
checkpoint — tanpa forward pass. Bentuknya dicetak supaya ketidakcocokan kelas
ketahuan di sini, bukan di tengah grid.

In [ ]:
import numpy as np, torch
HEAD_DIR = '/content/heads'
os.makedirs(HEAD_DIR, exist_ok=True)
HEADS = {}

def _save(tag, W, b):
    wp, bp = f'{HEAD_DIR}/{tag}_w.npy', f'{HEAD_DIR}/{tag}_b.npy'
    np.save(wp, np.asarray(W, dtype=np.float64))
    np.save(bp, np.zeros(len(W)) if b is None else np.asarray(b, dtype=np.float64))
    HEADS[tag] = (wp, bp)
    print(' ', tag, W.shape)

if not os.path.exists(f'{HEAD_DIR}/head_rn50_w.npy'):
    from torchvision.models import ResNet50_Weights, resnet50
    m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    _save('head_rn50', m.fc.weight.detach().cpu().numpy(),
          m.fc.bias.detach().cpu().numpy())
    del m
else:
    HEADS['head_rn50'] = (f'{HEAD_DIR}/head_rn50_w.npy', f'{HEAD_DIR}/head_rn50_b.npy')
    print('  head_rn50 sudah ada')

if not os.path.exists(f'{HEAD_DIR}/head_vit_w.npy'):
    from torchvision.models import ViT_B_16_Weights, vit_b_16
    m = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
    h = m.heads.head
    _save('head_vit', h.weight.detach().cpu().numpy(),
          h.bias.detach().cpu().numpy() if h.bias is not None else None)
    del m
else:
    HEADS['head_vit'] = (f'{HEAD_DIR}/head_vit_w.npy', f'{HEAD_DIR}/head_vit_b.npy')
    print('  head_vit sudah ada')

for t, (wp, _) in HEADS.items():
    W = np.load(wp, mmap_mode='r')
    assert W.shape[0] == 1000, t + ' bukan 1000 kelas: ' + str(W.shape)
print('kepala siap:', sorted(HEADS))

## 4. Logit → softmax `.npy`, dengan suhu Platt mereka

Suhu diambil dari manifest per metode bila tersedia (`temperatures`), kalau tidak
dari `temperature` tunggal. Kalau tidak ada sama sekali, dipakai 1.0 **dan dicetak
sebagai peringatan** — memakai suhu yang salah diam-diam akan menghasilkan skor yang
bukan skor mereka.

In [ ]:
import torch
os.makedirs(WORK, exist_ok=True)

def temp_for(man, method):
    ts = man.get('temperatures')
    if isinstance(ts, dict) and method in ts:
        return float(ts[method]), 'temperatures[' + method + ']'
    if man.get('temperature') is not None:
        return float(man['temperature']), 'temperature (tunggal)'
    return 1.0, 'TIDAK ADA -- dipakai 1.0, PERINGATAN'

PAIRS = {}
for s in SEEDS:
    sd = f'{CACHE}/seed_{s}'
    man = json.load(open(sd + '/manifest.json'))
    for m in METHODS:
        T, src = temp_for(man, m)
        out = {}
        for split in ('cal', 'test'):
            dst_s = f'{WORK}/{m}_s{s}_{split}_softmax.npy'
            dst_y = f'{WORK}/{m}_s{s}_{split}_labels.npy'
            if not (os.path.exists(dst_s) and os.path.exists(dst_y)):
                d = torch.load(f'{sd}/{m}_{split}.pt', map_location='cpu')
                z = d['logits'].to(torch.float64) / T
                p = torch.softmax(z, dim=1).numpy().astype(np.float32)
                np.save(dst_s, p)
                np.save(dst_y, d['labels'].numpy().astype(np.int64))
                del d, z, p
            out[split] = (dst_s, dst_y)
        PAIRS[(m, s)] = out
        if s == SEEDS[0]:
            print(f'  {m:14s} T={T:.4f}  ({src})')

k0 = (METHODS[0], SEEDS[0])
a = np.load(PAIRS[k0]['cal'][0], mmap_mode='r')
b = np.load(PAIRS[k0]['test'][0], mmap_mode='r')
print('cal', a.shape, '| test', b.shape)
row = np.asarray(a[:200]).sum(axis=1)
assert np.allclose(row, 1.0, atol=1e-3), 'baris tidak berjumlah 1 -- softmax gagal'
yv = np.load(PAIRS[k0]['test'][1])
acc = float((np.asarray(b).argmax(1) == yv).mean())
print('akurasi top-1 test:', round(acc, 4), '(ResNet-50 ImageNet ~0.76-0.81)')
assert 0.6 < acc < 0.9, 'akurasi di luar rentang wajar -- periksa suhu/label'
print('softmax terverifikasi')

## 5. Grid dan jalankan

3 keluarga φ × 4 metode agregasi × 2 α × 2 n_cal × 3 seed. DESC+CAL dari dump `cal`,
EVAL dari **seluruh** dump `test` — pemisahan yang sama seperti dump LTC.

In [ ]:
from pcc.experiments import phase2_pcc as drv
from pcc.utils.io import write_report
import traceback

PHIS = (('head_rn50', 'w_cos_knn_1'), ('head_vit', 'w_cos_knn_1'),
        ('output', 'prof_knn_1'))

class A: pass

def build(phi, hold, m, s, a, nc):
    p = PAIRS[(m, s)]
    x = A()
    x.scores, x.labels = p['cal']
    x.eval_scores, x.eval_labels = p['test']
    x.max_rows = None
    x.dataset = f'umtta_rn50_{m}'
    x.reports_dir = 'pcc/reports'
    x.alpha, x.n_cal = a, nc
    x.heldout_frac = HELDOUT_FRAC
    x.frac_desc, x.frac_cal = 0.40, 0.30
    x.phi = 'head' if phi.startswith('head') else 'output'
    x.head_weights = HEADS[phi][0] if x.phi == 'head' else None
    x.head_bias = HEADS[phi][1] if x.phi == 'head' else None
    x.distance_holdout = hold
    x.stat = 'worst'
    x.ccc_root = '/content/ccc' if os.path.isdir('/content/ccc') else None
    x.seed = s
    x.name = None
    x.print_json = False
    return x

GRID = [(phi, hold, m, s, a, nc)
        for phi, hold in PHIS for m in METHODS
        for a in ALPHAS for nc in N_CALS for s in SEEDS
        if (a == ALPHAS[0] and nc == N_CALS[0]) or s == SEEDS[0]]
print('konfigurasi:', len(GRID))

RES, FAIL = [], []
t_start = time.time()
for i, (phi, hold, m, s, a, nc) in enumerate(GRID, 1):
    tag = f'{phi}|{m}|a{a}|nc{nc}|s{s}'
    print(f'[{i}/{len(GRID)}] {tag}', flush=True)
    t0 = time.time()
    try:
        x = build(phi, hold, m, s, a, nc)
        r = drv.run(x)
        c = drv.verdict(r, x.stat)
        write_report('pcc/reports', f'nb08_{phi}_{m}_a{a}_nc{nc}_s{s}',
                     hypothesis=drv.HYPOTHESIS, pass_criteria=drv.PASS_CRITERIA,
                     config=vars(x), seed=s, results=r, conclusion=c,
                     started_at=t0)
        RES.append(dict(phi=phi, method=m, alpha=a, n_cal=nc, seed=s,
                        res=r, conclusion=c,
                        headline=(a == ALPHAS[0] and nc == N_CALS[0])))
        t1, t2 = r['table_1_seen'], r.get('table_2_heldout')
        s1 = t1['primary_stat']
        msg = '    {:.0f}s | lam {:.3f} | T1[{}] {:+.4f}'.format(
            time.time()-t0, r['pcc']['lambda'], s1, t1['delta'].get(s1, float('nan')))
        if t2 is not None:
            s2 = t2['primary_stat']
            msg += ' | T2[{}] {:+.4f}'.format(s2, t2['delta'].get(s2, float('nan')))
        print(msg, '|', c, flush=True)
    except Exception as e:
        FAIL.append(dict(tag=tag, error=type(e).__name__ + ': ' + str(e)))
        print('    GAGAL:', FAIL[-1]['error'][:180], flush=True)
        traceback.print_exc()
print()
print('selesai {}/{} dalam {:.0f}s'.format(len(RES), len(GRID), time.time()-t_start))
for f in FAIL:
    print(' ', f['tag'], f['error'][:120])

## 6. Agregasi — dan pertanyaan yang dijawab notebook ini

Yang dibaca bukan cuma lulus/gagal. Tiga hal:

1. **`head_vit` vs `head_rn50`** — apakah klaim butuh φ dari model LAIN, atau
   model yang sama pun cukup? Ini pertanyaan reviewer yang pasti muncul.
2. **`head_*` vs `output`** — apakah pembalikan peringkat R² terulang di backbone
   kedua, atau khas CCC saja?
3. **Lintas metode agregasi** — apakah PCC bergantung pada kualitas skornya?

In [ ]:
from pcc.eval.stats import mean_ci
from collections import defaultdict

agg = defaultdict(lambda: defaultdict(list))
for r in RES:
    if not r['headline']:
        continue
    k = (r['phi'], r['method'])
    for tn in ('table_1_seen', 'table_2_heldout'):
        tb = r['res'].get(tn)
        if tb is None:
            continue
        st = tb['primary_stat']
        if st in tb['delta']:
            agg[k][tn + '|' + st].append(tb['delta'][st])
        agg[k][tn + '|macro'].append(tb['delta']['macro'])
        agg[k][tn + '|size_matched'].append(1.0 if tb['size_matched'] else 0.0)
    agg[k]['lambda'].append(r['res']['pcc']['lambda'])

SUMMARY = {}
print('=== CI antar-seed ===')
for k in sorted(agg):
    print('  ' + '|'.join(k))
    row = {}
    for met in sorted(agg[k]):
        ci = mean_ci(np.array(agg[k][met], float))
        row[met] = ci
        if met.endswith('size_matched'):
            if ci['mean'] < 1.0:
                print('      {:26s} UKURAN TIDAK COCOK sebagian seed'.format(met))
            continue
        flag = '  <- CI di atas 0' if (met.startswith('table_') and
                                      ci['ci_low'] > 0) else ''
        print('      {:26s} {:+.4f} [{:+.4f}, {:+.4f}]{}'.format(
            met, ci['mean'], ci['ci_low'], ci['ci_high'], flag))
    SUMMARY['|'.join(k)] = row

## 7. Laporan + simpan ke Drive

In [ ]:
CAVEATS = [
    'Backbone: torchvision ResNet-50 (UMTTA). Skor dari logit ter-cache + suhu Platt',
    '  dari manifest -- BUKAN softmax mentah.',
    'head_rn50 TIDAK eksogen di sini: model yang sama dengan sumber skornya.',
    '  head_vit yang eksogen. Keduanya dilaporkan sebagai dua tingkat klaim.',
    'Seed 3, bukan 10. alpha dan skor gratis (Phase B numpy), seed tidak.',
    'Sec 7 belum terpenuhi: baseline belum direproduksi lawan angka terbit.',
]
for c in CAVEATS:
    print('CAVEAT:', c)

path = write_report('pcc/reports', '08_pcc_on_umtta',
                    hypothesis=drv.HYPOTHESIS, pass_criteria=drv.PASS_CRITERIA,
                    config={'cache': CACHE, 'methods': list(METHODS),
                            'seeds': list(SEEDS), 'alphas': list(ALPHAS),
                            'n_cals': list(N_CALS), 'phis': [p[0] for p in PHIS],
                            'grid_size': len(GRID), 'seed': SEED},
                    seed=SEED,
                    results={'summary': SUMMARY, 'n_ok': len(RES),
                             'n_failed': len(FAIL), 'failed': FAIL,
                             'caveats': CAVEATS},
                    conclusion='SELESAI' if not FAIL else 'SEBAGIAN',
                    started_at=t_start)
print('laporan:', path)

import shutil
DEST = f"{DRIVE_ROOT}/runs/nb08_{time.strftime('%Y%m%d_%H%M%S')}"
os.makedirs(DEST, exist_ok=True)
src = sorted(glob.glob('pcc/reports/*.json'))
bad = []
for p in src:
    d = os.path.join(DEST, os.path.basename(p))
    shutil.copy2(p, d)
    if os.path.getsize(d) != os.path.getsize(p):
        bad.append(p)
z = shutil.make_archive(DEST, 'zip', DEST)
print('tersalin {}/{} -> {}'.format(len(src)-len(bad), len(src), DEST))
print('zip', z, '{:.1f} MB'.format(os.path.getsize(z)/1e6))
print('AMAN' if not bad else 'GAGAL: ' + str(bad[:3]))